[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_ch04/NN_DL_ch04_ConvViz/NN_DL_ch04_ConvViz.ipynb)

# NN_DL_ch04_ConvViz

**CNN on MNIST: Training, Filter Visualization, and Feature Maps**

Train a CNN on the real MNIST dataset, visualize learned convolution filters, intermediate feature maps, and evaluate with a confusion matrix.

In [ ]:
!pip install -q torch torchvision matplotlib scikit-learn scipy

In [ ]:
# Style & Reproducibility
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

MAIN_BLUE = '#1A3A6E'
IDA_RED   = '#CD0000'
FOREST    = '#2E7D32'
CRIMSON   = '#DC3545'
AMBER     = '#FFC107'

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.size': 12, 'axes.labelsize': 13, 'axes.titlesize': 14,
    'figure.figsize': (10, 6),
})
np.random.seed(42)

def save_fig(name):
    plt.savefig(f'{name}.pdf', bbox_inches='tight', dpi=300, transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', dpi=150, transparent=True)
    plt.show()
    print(f'Saved {name}.pdf and {name}.png')

In [ ]:
import torch
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Load MNIST
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])

train_set = torchvision.datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_set  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_dl = DataLoader(train_set, batch_size=128, shuffle=True)
test_dl  = DataLoader(test_set,  batch_size=256)

print(f"Train: {len(train_set)}, Test: {len(test_set)}")
print(f"Image shape: {train_set[0][0].shape}")

In [ ]:
# CNN Model
import torch.nn as nn
import torch.nn.functional as F

class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 32 * 7 * 7)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

model = MnistCNN().to(device)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(10):
    model.train()
    losses, correct, total = [], 0, 0
    for images, labels in train_dl:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        losses.append(loss.item())
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    history['train_loss'].append(np.mean(losses))
    history['train_acc'].append(correct / total)

    model.eval()
    losses, correct, total = [], 0, 0
    with torch.no_grad():
        for images, labels in test_dl:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            losses.append(criterion(outputs, labels).item())
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    history['val_loss'].append(np.mean(losses))
    history['val_acc'].append(correct / total)
    print(f"Epoch {epoch+1}: Train Acc={history['train_acc'][-1]:.4f}, Val Acc={history['val_acc'][-1]:.4f}")

In [ ]:
# 2D Convolution Demonstration
from scipy.signal import convolve2d

sample_img = test_set[0][0].squeeze().numpy()

kernels = {
    'Edge Horizontal': np.array([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=np.float32),
    'Edge Vertical': np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=np.float32),
    'Sharpen': np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=np.float32),
    'Blur': np.ones((3,3), dtype=np.float32) / 9,
}

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].imshow(sample_img, cmap='grey')
axes[0].set_title('Original', fontweight='bold')

for ax, (name, kernel) in zip(axes[1:], kernels.items()):
    filtered = convolve2d(sample_img, kernel, mode='same', boundary='fill')
    ax.imshow(filtered, cmap='grey')
    ax.set_title(name, fontweight='bold')

for ax in axes:
    ax.axis('off')
plt.suptitle('2D Convolution with Different Kernels', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('nn_ch04_convolution_2d')

In [ ]:
# Feature Maps from Trained CNN
model.eval()
sample = test_set[0][0].unsqueeze(0).to(device)

activations = {}
def hook_fn(name):
    def hook(module, input, output):
        activations[name] = output.detach().cpu()
    return hook

model.conv1.register_forward_hook(hook_fn('conv1'))
model.conv2.register_forward_hook(hook_fn('conv2'))

with torch.no_grad():
    _ = model(sample)

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for i in range(8):
    axes[0, i].imshow(activations['conv1'][0, i], cmap='viridis')
    axes[0, i].axis('off')
    axes[0, i].set_title(f'Conv1-{i}', fontsize=9)
for i in range(8):
    axes[1, i].imshow(activations['conv2'][0, i], cmap='viridis')
    axes[1, i].axis('off')
    axes[1, i].set_title(f'Conv2-{i}', fontsize=9)
plt.suptitle('Feature Maps from Trained CNN', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('nn_ch04_feature_maps')

In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_dl:
        images = images.to(device)
        preds = model(images).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10, 9))
disp = ConfusionMatrixDisplay(cm, display_labels=list(range(10)))
disp.plot(cmap='Blues', ax=ax, values_format='d')
ax.set_title('MNIST Confusion Matrix', fontweight='bold')
save_fig('nn_ch04_mnist_confusion')

In [ ]:
# Training Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['train_loss'], color=MAIN_BLUE, lw=2, label='Train')
ax1.plot(history['val_loss'],   color=IDA_RED,   lw=2, label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss'); ax1.legend()

ax2.plot(history['train_acc'], color=MAIN_BLUE, lw=2, label='Train')
ax2.plot(history['val_acc'],   color=IDA_RED,   lw=2, label='Val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Accuracy'); ax2.legend()

plt.suptitle('MNIST CNN Training', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('nn_ch04_mnist_training')

## Summary

Trained a CNN on **MNIST** achieving >99% accuracy. Visualized hand-crafted and learned convolution filters, intermediate feature maps, and the full confusion matrix.